In [ ]:
import glob
import os
import sys

import gin
import IPython.display as ipd
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

gin.enter_interactive_mode()


@gin.configurable("libdamp")
def libdamp_entry_point(experiment=gin.REQUIRED, train_dataset=gin.REQUIRED, val_dataset=None, test_dataset=None):
    """Entrance function for specifying an experiment and dataset for training in the gin config

    Example usage in the gin config:
    ```
    torchdsp.experiment = @MyModel
    torchdsp.train_dataset = @MyTrainDataset
    torchdsp.val_dataset = @MyValDataset
    torchdsp.test_dataset = @MyTestDataset
    ```

    Validation and test datasets are optional.
    Attention: The function name conflicts with the module name, but this is only relevant in this file.
    """
    return experiment, train_dataset, val_dataset, test_dataset

In [ ]:
fs = 16000.0
exp_id = "AAT3_LMA_1M_OF_20261106_082646"

sys.path.append("..")

# load operative gin config
gin.clear_config()
with gin.unlock_config():
    gin.parse_config_file(os.path.join("../results", exp_id, "operative_config.gin"), skip_unknown=True)

# create model and datasets
model, train_dataset, val_dataset, test_dataset = libdamp_entry_point()

best_ckpt = glob.glob(os.path.join("../results", exp_id, "checkpoints/*best*.ckpt"))[0]
last_ckpt = os.path.join("../results", exp_id, "checkpoints/last.ckpt")

# load the best checkpoint and configure the weights of the model
ckpt = torch.load(best_ckpt, map_location="cpu")
sd = ckpt["state_dict"]

model.load_state_dict(sd)
# model = model.cuda()
_ = model.eval()

assert model.fs == fs, "Sampling rate mismatch"

In [ ]:
# plot training and validation loss
steps_per_epoch = 188  # for calculating the epoch average

log = pd.read_csv(os.path.join("../results", exp_id, "csv_log/metrics.csv"))
train_loss = log[log["train_loss_step"].notnull()]
train_loss_epoch = log[log["train_loss_epoch"].notnull()]
val_loss = log[log["val_loss"].notnull()]

idx = train_loss_epoch["train_loss_epoch"].argmin()
print(f"Best train loss: {train_loss_epoch['train_loss_epoch'].iloc[idx]:.4f} (epoch {train_loss_epoch['step'].iloc[idx] / steps_per_epoch:.0f})")
print(f"Best val loss: {val_loss['val_loss'].min():.4f}, last val los: {val_loss['val_loss'].iloc[-1]:.4f}")

plt.plot(train_loss["step"], train_loss["train_loss_step"], label="train")
plt.plot(train_loss_epoch["step"] - 0.5 * steps_per_epoch, train_loss_epoch["train_loss_epoch"], label="train (epoch average)")
plt.plot(val_loss["step"], val_loss["val_loss"], label="validation")
plt.legend()
plt.yscale("log")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.show()

In [ ]:
# load audio
x, _ = librosa.load("/home/sws/datasets/lm-ssd/LM-SSD/audio/SSD003_AA_LM_C0_1M_T3.wav", sr=fs)
y_ref, _ = librosa.load("/home/sws/datasets/lm-ssd/LM-SSD/audio/SSD003_AA_CM_C0_1M_T3.wav", sr=fs)

# choose a good excerpt and convert to Tensors in the right shape
start = 46.5
duration = 20
x = torch.Tensor(x)[None, int(start * fs):int((start + duration) * fs)]
y_ref = y_ref[int(start * fs):int((start + duration) * fs)]

# run model
y, signals = model(x, return_intermediates=True)
y = y.detach()

# visualize
H = 1024
N = 4096
gamma = 10

X = np.abs(librosa.stft(x[0].numpy(), n_fft=N, hop_length=H))
Y = np.abs(librosa.stft(y[0].numpy(), n_fft=N, hop_length=H))
Y_ref = np.abs(librosa.stft(y_ref, n_fft=N, hop_length=H))

t_ = np.arange(X.shape[1]) * H / fs + start
f_ = np.arange(X.shape[0]) * fs / N

_, axs = plt.subplots(3, 1, figsize=(8, 6), dpi=150, sharex=True)

axs[0].set_title("CM", size=7)
axs[0].pcolormesh(t_, f_ / 1000, np.log(1 + 5 * Y_ref), cmap="gray_r")
axs[0].set_ylim(0.05, 5.5)
axs[0].set_ylabel("Freq. (kHz)")

axs[1].set_title("LM", size=7)
axs[1].pcolormesh(t_, f_ / 1000, np.log(1 + 5 * X), cmap="gray_r")
axs[1].set_ylim(0.05, 5.5)
axs[1].set_ylabel("Freq. (kHz)")

axs[2].set_title("Reconstructed from LM", size=7)
axs[2].pcolormesh(t_, f_ / 1000, np.log(1 + 5 * Y), cmap="gray_r")
axs[2].set_ylim(0.05, 5.5)
axs[2].set_ylabel("Freq. (kHz)")

axs[-1].set_xlabel("Time (s)")

plt.tight_layout()

plt.show()

print("CM:")
ipd.display(ipd.Audio(np.repeat(y_ref[None,:], 2, axis=0), rate=fs))
print("LM:")
ipd.display(ipd.Audio(np.repeat(x.numpy(), 2, axis=0), rate=fs))
print("Reconstructed from LM:")
ipd.display(ipd.Audio(np.repeat(y.numpy(), 2, axis=0), rate=fs))